# Sampled Scatter Decoding Orchestrator

This notebook orchestrates one Slurm job per patient/resample for the scatter-cluster XGBoost decoder.

Workflow:
1. resolve per-patient inputs from `vad_new`
2. submit resamples `0,1,2` with hyperparameter search
3. build a consensus parameter set from those first three fits
4. submit resamples `3..19` using the consensus parameters
5. aggregate all per-resample outputs into one CSV and one PKL per patient

Important modeling updates:
- the decoder now reads semantic cluster labels from `embeddings/semantic_cluster_predictions.npy`
- permutation testing uses the requested null: shuffle training-set `X` only, keep `y_train` fixed, evaluate on the untouched test set


In [1]:
import json
import subprocess
from pathlib import Path

import dill as pickle
import numpy as np
import pandas as pd
pd.set_option('display.max_columns', 200)
pd.set_option('display.max_rows', 200)


In [2]:
# -- Config -----------------------------------------------------------------
PATIENTS = ['YEY', 'YEZ', 'YFA', 'YFB', 'YFC', 'YFD', 'YFE', 'YFF', 'YFG', 'YFI', 'YFK', 'YFL', 'YFM', 'YFN', 'YFP', 'YFQ', 'YFR', 'YFT', 'YFU', 'YFV']

VAD_ROOT = Path('/mnt/labworlds/Hayden/Hayden_Lab/speech_247/vad_new')
LEGACY_ROOT = Path('/mnt/labworlds/Hayden/Hayden_Lab/speech_247/vad_out')

CLUSTER_PREDS_OVERRIDE = {}
FRS_PATH_OVERRIDE = {}

N_RESAMPLES = 20
FIRST_STAGE = [0, 1, 2]
SECOND_STAGE = list(range(3, N_RESAMPLES))
SEED_STRIDE = 42

PYTHON_BIN = '/scratch/tahaismail424/miniforge3/envs/speech_247_env/bin/python3'
WORKER_PATH = Path('/scratch/tahaismail424/speech_247/notebooks/REBIRTH_2!/standard_decoding_analysis/scat_classifier_worker.py')

PARTITION = 'Universal'
QOS = 'default_tier'
CPUS = 8
MEM = '40G'
TIME = '48:00:00'

TEST_SIZE = 0.2
SEARCH_ITERS = 40
CV_SPLITS = 4
N_SHUFFLES = 50
SCORING = 'f1_macro'
PERM_METRIC = 'f1_macro'

print('patients:', PATIENTS)
print('worker:', WORKER_PATH)

patients: ['YEY', 'YEZ', 'YFA', 'YFB', 'YFC', 'YFD', 'YFE', 'YFF', 'YFG', 'YFI', 'YFK', 'YFL', 'YFM', 'YFN', 'YFP', 'YFQ', 'YFR', 'YFT', 'YFU', 'YFV']
worker: /scratch/tahaismail424/speech_247/notebooks/REBIRTH_2!/standard_decoding_analysis/scat_classifier_worker.py


In [3]:
# -- Path resolution helpers -------------------------------------------------
def first_existing(paths):
    for p in paths:
        if p is not None and Path(p).exists():
            return Path(p)
    return None


def patient_root(patient: str) -> Path:
    return VAD_ROOT / patient


def output_root(patient: str) -> Path:
    out = patient_root(patient) / 'standard_decoding_analysis' / 'scat_xgboost_sampled_norm'
    out.mkdir(parents=True, exist_ok=True)
    return out


def logs_dir(patient: str) -> Path:
    p = patient_root(patient) / 'decoding_slurm_logs' / 'scat_classifier_sampled_nocv'
    p.mkdir(parents=True, exist_ok=True)
    return p


def scripts_dir(patient: str) -> Path:
    p = patient_root(patient) / 'decoding_slurm_scripts' / 'scat_classifier_sampled_nocv'
    p.mkdir(parents=True, exist_ok=True)
    return p


def resolve_patient_inputs(patient: str) -> dict:
    root = patient_root(patient)
    cluster_override = CLUSTER_PREDS_OVERRIDE.get(patient)
    frs_override = FRS_PATH_OVERRIDE.get(patient)

    new_clusters = first_existing([root / 'embeddings' / 'semantic_cluster_predictions.npy'])
    new_frs = first_existing([
        root / 'neural_embeddings' / 'word_frs.npy',
        root / 'all_convo_recording' / 'word_avg_frs.npy',
    ])
    safe_df = first_existing([root / 'decoding_inputs' / 'all_words_filtered.csv'])
    legacy_filtered = first_existing([LEGACY_ROOT / patient / 'all_convo_recording' / 'all_words_filtered.csv'])
    legacy_frs = first_existing([LEGACY_ROOT / patient / 'all_convo_recording' / 'word_avg_frs.npy'])

    if cluster_override is not None or frs_override is not None:
        cluster_preds_path = first_existing([cluster_override, new_clusters])
        frs_path = first_existing([frs_override, new_frs, legacy_frs])
        input_source = 'override'
    elif new_clusters is not None and new_frs is not None:
        cluster_preds_path = new_clusters
        frs_path = new_frs
        input_source = 'vad_new'
    else:
        cluster_preds_path = None
        frs_path = None
        input_source = 'missing'

    out = output_root(patient)
    return {
        'patient': patient,
        'patient_root': root,
        'input_source': input_source,
        'cluster_preds_path': cluster_preds_path,
        'frs_path': frs_path,
        'safe_df': safe_df,
        'legacy_filtered': legacy_filtered,
        'output_root': out,
        'consensus_params_json': out / 'consensus_best_params.json',
        'aggregate_csv': out / 'scat_sampled_all_resamples.csv',
        'aggregate_pkl': out / 'scat_sampled_all_resamples.pkl',
        'logs_dir': logs_dir(patient),
        'scripts_dir': scripts_dir(patient),
    }


def choose_consensus_params(param_options: list[dict]) -> dict:
    params = dict(param_options[0])
    params['max_depth'] = int(np.min([p['max_depth'] for p in param_options]))
    params['min_child_weight'] = int(np.max([p['min_child_weight'] for p in param_options]))
    params['gamma'] = float(np.max([p['gamma'] for p in param_options]))
    params['reg_lambda'] = float(np.max([p['reg_lambda'] for p in param_options]))
    params['reg_alpha'] = float(np.max([p['reg_alpha'] for p in param_options]))
    params['subsample'] = float(np.median([p['subsample'] for p in param_options]))
    params['colsample_bytree'] = float(np.median([p['colsample_bytree'] for p in param_options]))
    params['learning_rate'] = float(np.median([p['learning_rate'] for p in param_options]))
    params['n_estimators'] = int(np.median([p['n_estimators'] for p in param_options]))
    params['tree_method'] = 'hist'
    params['device'] = 'cpu'
    params.pop('predictor', None)
    params.pop('use_label_encoder', None)
    return params

In [4]:
# -- Status helpers ----------------------------------------------------------
def resample_paths(out_root: Path, r: int) -> dict:
    return {
        'pkl': out_root / f'scat_sampled_resample_{r}_.pkl',
        'summary_json': out_root / f'summary_resample_{r}.json',
        'best_params_json': out_root / f'best_params_resample_{r}.json',
        'success': out_root / f'resample_{r}_SUCCESS',
        'error': out_root / f'resample_{r}_error.txt',
    }


def build_patient_status(patient: str) -> pd.DataFrame:
    info = resolve_patient_inputs(patient)
    rows = []
    for r in range(N_RESAMPLES):
        paths = resample_paths(info['output_root'], r)
        rows.append({
            'patient': patient,
            'resample_idx': r,
            'stage': 'hyperparam' if r in FIRST_STAGE else 'fixed',
            'seed': r * SEED_STRIDE,
            'cluster_preds_path': info['cluster_preds_path'],
            'frs_path': info['frs_path'],
            'safe_df': info['safe_df'],
            'has_inputs': info['cluster_preds_path'] is not None and info['frs_path'] is not None,
            'consensus_params_json': info['consensus_params_json'],
            'consensus_ready': info['consensus_params_json'].exists(),
            'pkl_path': paths['pkl'],
            'summary_json': paths['summary_json'],
            'best_params_json': paths['best_params_json'],
            'success_path': paths['success'],
            'error_path': paths['error'],
            'has_pkl': paths['pkl'].exists(),
            'has_summary': paths['summary_json'].exists(),
            'has_best_params': paths['best_params_json'].exists(),
            'has_success': paths['success'].exists(),
            'has_error': paths['error'].exists(),
        })
    df = pd.DataFrame(rows)
    first_three_done = bool(df[df['resample_idx'].isin(FIRST_STAGE)]['has_success'].all()) if not df.empty else False
    df['first_three_done'] = first_three_done
    df['ready_phase1'] = df['has_inputs'] & df['resample_idx'].isin(FIRST_STAGE) & ~df['has_success']
    df['ready_phase2'] = df['has_inputs'] & df['resample_idx'].isin(SECOND_STAGE) & first_three_done & df['consensus_ready'] & ~df['has_success']
    return df


status_df = pd.concat([build_patient_status(pt) for pt in PATIENTS], ignore_index=True)
status_df[['patient','resample_idx','stage','has_inputs','has_success','has_error','has_best_params','consensus_ready','ready_phase1','ready_phase2']]


,patient,resample_idx,stage,has_inputs,has_success,has_error,has_best_params,consensus_ready,ready_phase1,ready_phase2
0,YEY,0,hyperparam,True,True,False,True,True,False,False
1,YEY,1,hyperparam,True,True,False,True,True,False,False
2,YEY,2,hyperparam,True,True,False,True,True,False,False
3,YEY,3,fixed,True,True,False,True,True,False,False
4,YEY,4,fixed,True,True,False,True,True,False,False
...,...,...,...,...,...,...,...,...,...,...
395,YFV,15,fixed,True,True,False,True,True,False,False
396,YFV,16,fixed,True,True,False,True,True,False,False
397,YFV,17,fixed,True,True,False,True,True,False,False
398,YFV,18,fixed,True,True,False,True,True,False,False


In [5]:
# -- Patient input summary ---------------------------------------------------
patient_info_df = pd.DataFrame([resolve_patient_inputs(pt) for pt in PATIENTS])
patient_info_df[['patient', 'input_source', 'cluster_preds_path', 'frs_path', 'safe_df', 'output_root', 'consensus_params_json']]


,patient,input_source,cluster_preds_path,frs_path,safe_df,output_root,consensus_params_json
0,YEY,vad_new,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...
1,YEZ,vad_new,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...
2,YFA,vad_new,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...
3,YFB,vad_new,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...
4,YFC,vad_new,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...
5,YFD,vad_new,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...
6,YFE,vad_new,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...
7,YFF,vad_new,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...
8,YFG,vad_new,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...
9,YFI,vad_new,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...


In [6]:
    # -- Submit first 3 resamples per patient -----------------------------------
DRY_RUN = False
SUBMIT_ONE = False

submitted = []
failed = []

for patient in PATIENTS:
    info = resolve_patient_inputs(patient)
    patient_status = build_patient_status(patient)
    for _, row in patient_status[patient_status['ready_phase1']].iterrows():
        r = int(row['resample_idx'])
        sbatch_text = f'''#!/bin/bash
#SBATCH --job-name=scat_{patient}_{r}
#SBATCH --partition={PARTITION}
#SBATCH --qos={QOS}
#SBATCH --cpus-per-task={CPUS}
#SBATCH --mem={MEM}
#SBATCH --time={TIME}
#SBATCH --output={info['logs_dir']}/{patient}_r{r}_%j.out
#SBATCH --error={info['logs_dir']}/{patient}_r{r}_%j.err

set -eo pipefail

echo "HOSTNAME: $(hostname)"
echo "START: $(date)"
echo "PATIENT: {patient}"
echo "RESAMPLE: {r}"

{PYTHON_BIN} {WORKER_PATH}   --patient {patient}   --resample-idx {r}   --cluster-preds-path "{info['cluster_preds_path']}"   --frs-path "{info['frs_path']}"   --out-dir "{info['output_root']}"   --test-size {TEST_SIZE}   --n-iter {SEARCH_ITERS}   --cv-splits {CV_SPLITS}   --n-shuffles {N_SHUFFLES}   --seed-stride {SEED_STRIDE}   --n-jobs {CPUS * 2}   --scoring {SCORING}   --perm-metric {PERM_METRIC}

echo "END: $(date)"
'''
        sbatch_path = info['scripts_dir'] / f'{patient}_resample_{r}.sbatch'
        sbatch_path.write_text(sbatch_text)
        if DRY_RUN:
            submitted.append((patient, r, 'dry-run'))
            print(f'dry-run: {patient} r={r}')
        else:
            try:
                res = subprocess.run(['sbatch', str(sbatch_path)], capture_output=True, text=True, check=True)
                submitted.append((patient, r, res.stdout.strip()))
                print(f'submitted: {patient} r={r} -> {res.stdout.strip()}')
            except subprocess.CalledProcessError as exc:
                failed.append((patient, r, exc.stderr.strip()))
                print(f'FAILED SUBMISSION: {patient} r={r}')
                print(exc.stderr)
        if SUBMIT_ONE and submitted:
            break
    if SUBMIT_ONE and submitted:
        break

print(f'submitted={len(submitted)}, failed={len(failed)}')

submitted: YEY r=0 -> Submitted batch job 224885
submitted: YEY r=1 -> Submitted batch job 224886
submitted: YEY r=2 -> Submitted batch job 224887
submitted: YEZ r=0 -> Submitted batch job 224888
submitted: YEZ r=1 -> Submitted batch job 224889
submitted: YEZ r=2 -> Submitted batch job 224890
submitted: YFA r=0 -> Submitted batch job 224891
submitted: YFA r=1 -> Submitted batch job 224892
submitted: YFA r=2 -> Submitted batch job 224893
submitted=9, failed=0


In [6]:
# -- Build consensus params once resamples 0..2 finish ----------------------
built = []
blocked = []
for patient in PATIENTS:
    info = resolve_patient_inputs(patient)
    patient_status = build_patient_status(patient)
    if info['consensus_params_json'].exists():
        print(f"consensus already exists for {patient}: {info['consensus_params_json']}")
        continue
    first_three = patient_status[patient_status['resample_idx'].isin(FIRST_STAGE)]
    if not bool(first_three['has_success'].all()):
        blocked.append(patient)
        print(f'waiting on first three for {patient}')
        continue
    param_options = []
    for r in FIRST_STAGE:
        path = info['output_root'] / f'best_params_resample_{r}.json'
        param_options.append(json.loads(path.read_text()))
    consensus = choose_consensus_params(param_options)
    info['consensus_params_json'].write_text(json.dumps(consensus, indent=2))
    built.append(patient)
    print(f"built consensus params for {patient} -> {info['consensus_params_json']}")

print('built:', built)
print('blocked:', blocked)


consensus already exists for YEY: /mnt/labworlds/Hayden/Hayden_Lab/speech_247/vad_new/YEY/standard_decoding_analysis/scat_xgboost_sampled_norm/consensus_best_params.json
consensus already exists for YEZ: /mnt/labworlds/Hayden/Hayden_Lab/speech_247/vad_new/YEZ/standard_decoding_analysis/scat_xgboost_sampled_norm/consensus_best_params.json
consensus already exists for YFA: /mnt/labworlds/Hayden/Hayden_Lab/speech_247/vad_new/YFA/standard_decoding_analysis/scat_xgboost_sampled_norm/consensus_best_params.json
consensus already exists for YFB: /mnt/labworlds/Hayden/Hayden_Lab/speech_247/vad_new/YFB/standard_decoding_analysis/scat_xgboost_sampled_norm/consensus_best_params.json
consensus already exists for YFC: /mnt/labworlds/Hayden/Hayden_Lab/speech_247/vad_new/YFC/standard_decoding_analysis/scat_xgboost_sampled_norm/consensus_best_params.json
consensus already exists for YFD: /mnt/labworlds/Hayden/Hayden_Lab/speech_247/vad_new/YFD/standard_decoding_analysis/scat_xgboost_sampled_norm/consens

In [7]:
    # -- Submit remaining 17 resamples after consensus exists -------------------
DRY_RUN = False
SUBMIT_ONE = False

submitted = []
failed = []

for patient in PATIENTS:
    info = resolve_patient_inputs(patient)
    patient_status = build_patient_status(patient)
    for _, row in patient_status[patient_status['ready_phase2']].iterrows():
        r = int(row['resample_idx'])
        sbatch_text = f'''#!/bin/bash
#SBATCH --job-name=scat_{patient}_{r}
#SBATCH --partition={PARTITION}
#SBATCH --qos={QOS}
#SBATCH --cpus-per-task={CPUS}
#SBATCH --mem={MEM}
#SBATCH --time={TIME}
#SBATCH --output={info['logs_dir']}/{patient}_r{r}_%j.out
#SBATCH --error={info['logs_dir']}/{patient}_r{r}_%j.err

set -eo pipefail

echo "HOSTNAME: $(hostname)"
echo "START: $(date)"
echo "PATIENT: {patient}"
echo "RESAMPLE: {r}"

{PYTHON_BIN} {WORKER_PATH}   --patient {patient}   --resample-idx {r}   --cluster-preds-path "{info['cluster_preds_path']}"   --frs-path "{info['frs_path']}"   --out-dir "{info['output_root']}"   --params-json "{info['consensus_params_json']}"   --test-size {TEST_SIZE}   --n-iter {SEARCH_ITERS}   --cv-splits {CV_SPLITS}   --n-shuffles {N_SHUFFLES}   --seed-stride {SEED_STRIDE}   --n-jobs {CPUS * 2}   --scoring {SCORING}   --perm-metric {PERM_METRIC}

echo "END: $(date)"
'''
        sbatch_path = info['scripts_dir'] / f'{patient}_resample_{r}.sbatch'
        sbatch_path.write_text(sbatch_text)
        if DRY_RUN:
            submitted.append((patient, r, 'dry-run'))
            print(f'dry-run: {patient} r={r}')
        else:
            try:
                res = subprocess.run(['sbatch', str(sbatch_path)], capture_output=True, text=True, check=True)
                submitted.append((patient, r, res.stdout.strip()))
                print(f'submitted: {patient} r={r} -> {res.stdout.strip()}')
            except subprocess.CalledProcessError as exc:
                failed.append((patient, r, exc.stderr.strip()))
                print(f'FAILED SUBMISSION: {patient} r={r}')
                print(exc.stderr)
        if SUBMIT_ONE and submitted:
            break
    if SUBMIT_ONE and submitted:
        break

print(f'submitted={len(submitted)}, failed={len(failed)}')

submitted=0, failed=0


In [8]:
# -- Error inspection --------------------------------------------------------
for patient in PATIENTS:
    patient_status = build_patient_status(patient)
    errs = patient_status[patient_status['has_error']]
    if errs.empty:
        print(f'no error files for {patient}')
        continue
    print(f'errors for {patient}:')
    display(errs[['resample_idx', 'error_path']])


no error files for YEY
no error files for YEZ
no error files for YFA
no error files for YFB
no error files for YFC
no error files for YFD
no error files for YFE
no error files for YFF
no error files for YFG
no error files for YFI
no error files for YFK
no error files for YFL
no error files for YFM
no error files for YFN
no error files for YFP
no error files for YFQ
no error files for YFR
no error files for YFT
no error files for YFU
no error files for YFV


In [9]:
# -- Aggregate full patient results once all 20 finish ----------------------
for patient in PATIENTS:
    info = resolve_patient_inputs(patient)
    patient_status = build_patient_status(patient)
    if not bool(patient_status['has_success'].all()):
        print(f'skipping aggregate for {patient}: not all resamples finished')
        continue

    summary_rows = []
    aggregate = {}
    for r in range(N_RESAMPLES):
        with open(info['output_root'] / f'summary_resample_{r}.json') as f:
            summary_rows.append(json.load(f))
        with open(info['output_root'] / f'scat_sampled_resample_{r}_.pkl', 'rb') as f:
            aggregate[f'resample_{r}'] = pickle.load(f)

    summary_df = pd.DataFrame(summary_rows).sort_values('resample_idx').reset_index(drop=True)
    summary_df.to_csv(info['aggregate_csv'], index=False)
    with open(info['aggregate_pkl'], 'wb') as f:
        pickle.dump(aggregate, f)

    print(f'aggregated {patient}')
    print(f"  csv: {info['aggregate_csv']}")
    print(f"  pkl: {info['aggregate_pkl']}")
    display(summary_df)


aggregated YEY
  csv: /mnt/labworlds/Hayden/Hayden_Lab/speech_247/vad_new/YEY/standard_decoding_analysis/scat_xgboost_sampled_norm/scat_sampled_all_resamples.csv
  pkl: /mnt/labworlds/Hayden/Hayden_Lab/speech_247/vad_new/YEY/standard_decoding_analysis/scat_xgboost_sampled_norm/scat_sampled_all_resamples.pkl


,patient,resample_idx,seed,n_sampled_raw,n_sampled_final,n_test,acc,balanced_accuracy,f1_macro,f1_micro,auc_macro_ovr,baseline_proportional_acc,baseline_majority_acc,binomial_p_vs_proportional,perm_metric,observed_perm_metric,perm_pvalue,cv_best_score,perm_null
0,YEY,0,0,10453,10453,2091,0.174079,0.135540,0.130116,0.174079,0.573950,0.1192,0.135342,1.529662e-13,f1_macro,0.130116,0.019608,0.130672,shuffle_training_X_only
1,YEY,1,42,10453,10453,2091,0.187470,0.141841,0.130116,0.187470,0.574979,0.1192,0.135342,1.475548e-19,f1_macro,0.130116,0.019608,0.130225,shuffle_training_X_only
2,YEY,2,84,10453,10453,2091,0.173123,0.135669,0.131167,0.173123,0.571769,0.1192,0.135342,3.736641e-13,f1_macro,0.131167,0.019608,0.127531,shuffle_training_X_only
3,YEY,3,126,10453,10453,2091,0.170732,0.130430,0.119941,0.170732,0.580791,0.1192,0.135342,3.290606e-12,f1_macro,0.119941,0.019608,NaN,shuffle_training_X_only
4,YEY,4,168,10453,10453,2091,0.186035,0.138535,0.122372,0.186035,0.569905,0.1192,0.135342,7.326176e-19,f1_macro,0.122372,0.019608,NaN,shuffle_training_X_only
5,YEY,5,210,10453,10453,2091,0.181253,0.138155,0.126041,0.181253,0.575968,0.1192,0.135342,1.248432e-16,f1_macro,0.126041,0.019608,NaN,shuffle_training_X_only
6,YEY,6,252,10453,10453,2091,0.165471,0.123757,0.110367,0.165471,0.563678,0.1192,0.135342,2.944998e-10,f1_macro,0.110367,0.019608,NaN,shuffle_training_X_only
7,YEY,7,294,10453,10453,2091,0.174079,0.133046,0.123282,0.174079,0.564429,0.1192,0.135342,1.529662e-13,f1_macro,0.123282,0.019608,NaN,shuffle_training_X_only
8,YEY,8,336,10453,10453,2091,0.187948,0.141025,0.125336,0.187948,0.580072,0.1192,0.135342,8.595795e-20,f1_macro,0.125336,0.019608,NaN,shuffle_training_X_only
9,YEY,9,378,10453,10453,2091,0.181253,0.135502,0.118487,0.181253,0.578784,0.1192,0.135342,1.248432e-16,f1_macro,0.118487,0.019608,NaN,shuffle_training_X_only


aggregated YEZ
  csv: /mnt/labworlds/Hayden/Hayden_Lab/speech_247/vad_new/YEZ/standard_decoding_analysis/scat_xgboost_sampled_norm/scat_sampled_all_resamples.csv
  pkl: /mnt/labworlds/Hayden/Hayden_Lab/speech_247/vad_new/YEZ/standard_decoding_analysis/scat_xgboost_sampled_norm/scat_sampled_all_resamples.pkl


,patient,resample_idx,seed,n_sampled_raw,n_sampled_final,n_test,acc,balanced_accuracy,f1_macro,f1_micro,auc_macro_ovr,baseline_proportional_acc,baseline_majority_acc,binomial_p_vs_proportional,perm_metric,observed_perm_metric,perm_pvalue,cv_best_score,perm_null
0,YEZ,0,0,22484,22484,4497,0.214143,0.157457,0.152970,0.214143,0.610096,0.127886,0.145653,5.489229e-58,f1_macro,0.152970,0.019608,0.143700,shuffle_training_X_only
1,YEZ,1,42,22484,22484,4497,0.210585,0.153788,0.148770,0.210585,0.611373,0.127886,0.145653,9.662717e-54,f1_macro,0.148770,0.019608,0.144387,shuffle_training_X_only
2,YEZ,2,84,22484,22484,4497,0.215255,0.154704,0.145776,0.215255,0.612695,0.127886,0.145653,2.413752e-59,f1_macro,0.145776,0.019608,0.146735,shuffle_training_X_only
3,YEZ,3,126,22484,22484,4497,0.214588,0.151840,0.137977,0.214588,0.614948,0.127886,0.145653,1.579409e-58,f1_macro,0.137977,0.019608,NaN,shuffle_training_X_only
4,YEZ,4,168,22484,22484,4497,0.211474,0.151647,0.142111,0.211474,0.611192,0.127886,0.145653,8.659793e-55,f1_macro,0.142111,0.019608,NaN,shuffle_training_X_only
5,YEZ,5,210,22484,22484,4497,0.213253,0.151987,0.140463,0.213253,0.615419,0.127886,0.145653,6.526588e-57,f1_macro,0.140463,0.019608,NaN,shuffle_training_X_only
6,YEZ,6,252,22484,22484,4497,0.217256,0.156347,0.146941,0.217256,0.606174,0.127886,0.145653,8.027787e-62,f1_macro,0.146941,0.019608,NaN,shuffle_training_X_only
7,YEZ,7,294,22484,22484,4497,0.222815,0.158801,0.148108,0.222815,0.607215,0.127886,0.145653,6.047558e-69,f1_macro,0.148108,0.019608,NaN,shuffle_training_X_only
8,YEZ,8,336,22484,22484,4497,0.209251,0.150959,0.142947,0.209251,0.617320,0.127886,0.145653,3.460474e-52,f1_macro,0.142947,0.019608,NaN,shuffle_training_X_only
9,YEZ,9,378,22484,22484,4497,0.214365,0.153513,0.141775,0.214365,0.615003,0.127886,0.145653,2.946378e-58,f1_macro,0.141775,0.019608,NaN,shuffle_training_X_only


aggregated YFA
  csv: /mnt/labworlds/Hayden/Hayden_Lab/speech_247/vad_new/YFA/standard_decoding_analysis/scat_xgboost_sampled_norm/scat_sampled_all_resamples.csv
  pkl: /mnt/labworlds/Hayden/Hayden_Lab/speech_247/vad_new/YFA/standard_decoding_analysis/scat_xgboost_sampled_norm/scat_sampled_all_resamples.pkl


...

,patient,resample_idx,seed,n_sampled_raw,n_sampled_final,n_test,acc,balanced_accuracy,f1_macro,f1_micro,auc_macro_ovr,baseline_proportional_acc,baseline_majority_acc,binomial_p_vs_proportional,perm_metric,observed_perm_metric,perm_pvalue,cv_best_score,perm_null
0,YFT,0,0,139415,129119,25824,0.219950,0.165347,0.151126,0.219950,0.626495,0.121254,0.138476,0.0,f1_macro,0.151126,0.019608,0.151395,shuffle_training_X_only
1,YFT,1,42,139415,129069,25814,0.221856,0.168782,0.157438,0.221856,0.625875,0.121242,0.138491,0.0,f1_macro,0.157438,0.019608,0.151910,shuffle_training_X_only
2,YFT,2,84,139415,129133,25827,0.217640,0.164434,0.151802,0.217640,0.622768,0.121258,0.138653,0.0,f1_macro,0.151802,0.019608,0.155006,shuffle_training_X_only
3,YFT,3,126,139415,128933,25787,0.221895,0.168032,0.155612,0.221895,0.627319,0.121209,0.138675,0.0,f1_macro,0.155612,0.019608,NaN,shuffle_training_X_only
4,YFT,4,168,139415,128968,25794,0.217764,0.164848,0.152454,0.217764,0.625594,0.121217,0.138521,0.0,f1_macro,0.152454,0.019608,NaN,shuffle_training_X_only
5,YFT,5,210,139415,129116,25824,0.223590,0.168632,0.155193,0.223590,0.628550,0.121254,0.138476,0.0,f1_macro,0.155193,0.019608,NaN,shuffle_training_X_only
6,YFT,6,252,139415,129091,25819,0.220380,0.166414,0.153351,0.220380,0.627294,0.121248,0.138464,0.0,f1_macro,0.153351,0.019608,NaN,shuffle_training_X_only
7,YFT,7,294,139415,129146,25830,0.220132,0.166553,0.153429,0.220132,0.628593,0.121262,0.138482,0.0,f1_macro,0.153429,0.019608,NaN,shuffle_training_X_only
8,YFT,8,336,139415,129080,25816,0.223311,0.169390,0.156696,0.223311,0.627424,0.121244,0.138403,0.0,f1_macro,0.156696,0.019608,NaN,shuffle_training_X_only
9,YFT,9,378,139415,129041,25809,0.221551,0.166837,0.152959,0.221551,0.628744,0.121236,0.138479,0.0,f1_macro,0.152959,0.019608,NaN,shuffle_training_X_only


aggregated YFU
  csv: /mnt/labworlds/Hayden/Hayden_Lab/speech_247/vad_new/YFU/standard_decoding_analysis/scat_xgboost_sampled_norm/scat_sampled_all_resamples.csv
  pkl: /mnt/labworlds/Hayden/Hayden_Lab/speech_247/vad_new/YFU/standard_decoding_analysis/scat_xgboost_sampled_norm/scat_sampled_all_resamples.pkl


,patient,resample_idx,seed,n_sampled_raw,n_sampled_final,n_test,acc,balanced_accuracy,f1_macro,f1_micro,auc_macro_ovr,baseline_proportional_acc,baseline_majority_acc,binomial_p_vs_proportional,perm_metric,observed_perm_metric,perm_pvalue,cv_best_score,perm_null
0,YFU,0,0,116523,113255,22651,0.216194,0.164001,0.148026,0.216194,0.617688,0.119468,0.137698,0.0,f1_macro,0.148026,0.019608,0.149776,shuffle_training_X_only
1,YFU,1,42,116523,113244,22649,0.216963,0.165546,0.151396,0.216963,0.614571,0.119465,0.137710,0.0,f1_macro,0.151396,0.019608,0.151180,shuffle_training_X_only
2,YFU,2,84,116523,113274,22655,0.219775,0.166744,0.149675,0.219775,0.620262,0.119474,0.137674,0.0,f1_macro,0.149675,0.019608,0.150062,shuffle_training_X_only
3,YFU,3,126,116523,113243,22649,0.217096,0.165536,0.151254,0.217096,0.620180,0.119465,0.137710,0.0,f1_macro,0.151254,0.019608,NaN,shuffle_training_X_only
4,YFU,4,168,116523,113267,22654,0.223139,0.169655,0.154132,0.223139,0.619804,0.119473,0.137592,0.0,f1_macro,0.154132,0.019608,NaN,shuffle_training_X_only
5,YFU,5,210,116523,113235,22647,0.217910,0.166959,0.153217,0.217910,0.621205,0.119462,0.137634,0.0,f1_macro,0.153217,0.019608,NaN,shuffle_training_X_only
6,YFU,6,252,116523,113283,22657,0.218476,0.165878,0.151132,0.218476,0.619951,0.119477,0.137573,0.0,f1_macro,0.151132,0.019608,NaN,shuffle_training_X_only
7,YFU,7,294,116523,113201,22641,0.222252,0.168982,0.153419,0.222252,0.620381,0.119453,0.137715,0.0,f1_macro,0.153419,0.019608,NaN,shuffle_training_X_only
8,YFU,8,336,116523,113265,22653,0.220103,0.168277,0.154302,0.220103,0.619738,0.119471,0.137686,0.0,f1_macro,0.154302,0.019608,NaN,shuffle_training_X_only
9,YFU,9,378,116523,113291,22659,0.219692,0.167531,0.153400,0.219692,0.620697,0.119480,0.137649,0.0,f1_macro,0.153400,0.019608,NaN,shuffle_training_X_only


aggregated YFV
  csv: /mnt/labworlds/Hayden/Hayden_Lab/speech_247/vad_new/YFV/standard_decoding_analysis/scat_xgboost_sampled_norm/scat_sampled_all_resamples.csv
  pkl: /mnt/labworlds/Hayden/Hayden_Lab/speech_247/vad_new/YFV/standard_decoding_analysis/scat_xgboost_sampled_norm/scat_sampled_all_resamples.pkl


,patient,resample_idx,seed,n_sampled_raw,n_sampled_final,n_test,acc,balanced_accuracy,f1_macro,f1_micro,auc_macro_ovr,baseline_proportional_acc,baseline_majority_acc,binomial_p_vs_proportional,perm_metric,observed_perm_metric,perm_pvalue,cv_best_score,perm_null
0,YFV,0,0,35528,31703,6341,0.222520,0.165878,0.151369,0.222520,0.625932,0.121023,0.141145,7.311773e-113,f1_macro,0.151369,0.019608,0.150521,shuffle_training_X_only
1,YFV,1,42,35528,31691,6339,0.211390,0.161575,0.155191,0.211390,0.622093,0.121039,0.141032,2.498484e-91,f1_macro,0.155191,0.019608,0.149733,shuffle_training_X_only
2,YFV,2,84,35528,31677,6336,0.218277,0.163914,0.151452,0.218277,0.631454,0.120993,0.141098,1.951919e-104,f1_macro,0.151452,0.019608,0.148378,shuffle_training_X_only
3,YFV,3,126,35528,31712,6343,0.212360,0.158757,0.146555,0.212360,0.622876,0.121065,0.141258,4.080585e-93,f1_macro,0.146555,0.019608,NaN,shuffle_training_X_only
4,YFV,4,168,35528,31653,6331,0.224925,0.169763,0.159163,0.224925,0.630211,0.120967,0.141684,9.933967e-118,f1_macro,0.159163,0.019608,NaN,shuffle_training_X_only
5,YFV,5,210,35528,31732,6347,0.208130,0.157978,0.148789,0.208130,0.624324,0.121062,0.141642,1.761657e-85,f1_macro,0.148789,0.019608,NaN,shuffle_training_X_only
6,YFV,6,252,35528,31703,6341,0.211796,0.158639,0.146640,0.211796,0.620608,0.121054,0.141460,4.522069e-92,f1_macro,0.146640,0.019608,NaN,shuffle_training_X_only
7,YFV,7,294,35528,31640,6328,0.220607,0.165288,0.151451,0.220607,0.622192,0.120982,0.141909,6.449575e-109,f1_macro,0.151451,0.019608,NaN,shuffle_training_X_only
8,YFV,8,336,35528,31672,6335,0.217837,0.162600,0.149509,0.217837,0.618640,0.121019,0.141752,1.684520e-103,f1_macro,0.149509,0.019608,NaN,shuffle_training_X_only
9,YFV,9,378,35528,31648,6330,0.220063,0.165175,0.153478,0.220063,0.625190,0.120957,0.141232,6.173420e-108,f1_macro,0.153478,0.019608,NaN,shuffle_training_X_only
